<hr style="border:none;height:6px;background:#fff;margin:1em 0;">


<div style="text-align: center;">
  <h1>Dividend Taxation and Top-Income Inequality</h1>
  <h3>HEC Liege</h3>
  <h4><em>Lucas Dubois</em></h4>
</div>

<hr style="border:none;height:6px;background:#fff;margin:1em 0;">


<hr style="border:none;height:4px;background:#fff;margin:1em 0;">


In [59]:
import numpy as np
import pandas as pd
import cvxpy as cp
import matplotlib.pyplot as plt
path = '/Users/lucasdubois/Desktop/MASTERDATA/' 
df = pd.read_csv(path + "MASTER_FINAL.csv")
df = df.sort_values(["Country Name", "Year"])

In [60]:
inequality_variables = [
    "top10_inc",
    "top1_inc",
    "top10_w",
    "top1_w",
    "gini_disp",
    "gini_mkt",
    "gini_disp_se",
    "gini_mkt_se"
]

interaction_variables = [
    "log_rgdpe",
    "log_pop",
    "log_emp",
    "log_cn",
    "rgdpe_sq",
    "log_rgdpe_sq",
    "hc_ctfp",
    "labsh_irr",
    "cn_ctfp",
    "xr_irr",
    "csh_g_hc",
    "csh_c_hc",
    "rgdpe_ctfp",
    "pop_rgdpe",
    "delta_cn"
]

id_variables = ["Country Name", "Year"]

X = [
    col for col in df.columns
    if col not in inequality_variables + interaction_variables + id_variables
]

X_numeric = df[X].select_dtypes(include=["number"]).columns.tolist()

<hr style="border:none;height:4px;background:#fff;margin:1em 0;">


<div style="text-align: align;">
  <h2> <small>4</small>&nbsp;&nbsp;&nbsp;&nbsp; ASCM:</h2>
</div>

<hr style="border:none; border-top:2px dashed #fff; margin:1em 0;">

<div style="text-align: align;">
  <h3> <small>3.1</small>&nbsp;&nbsp;&nbsp;&nbsp;Preparing Brazil:</h3>
</div>

In [67]:
def scm_ascm(
    df,
    country,
    treat_year,
    covariates,
    ridge_lambda=1.0,
    outcome_weight=10000.0,
    covariate_weight=1.0,
):
    import numpy as np
    import pandas as pd
    import cvxpy as cp

    # -----------------------
    # 1) PREP DATA
    # -----------------------
    df = df.sort_values(["Country Name", "Year"]).copy()

    needed_cols = ["Country Name", "Year", "top1_inc"] + covariates
    df = df[needed_cols].copy()

    years = sorted(df["Year"].unique())
    pre_years = [y for y in years if y < treat_year]
    post_years = [y for y in years if y >= treat_year]

    # Outcome matrix
    Y = df.pivot(index="Year", columns="Country Name", values="top1_inc").sort_index()

    # Covariate matrices
    cov_mats = {
        cov: df.pivot(index="Year", columns="Country Name", values=cov).sort_index()
        for cov in covariates
    }

    # Keep only countries with complete pre-treatment outcome + covariate data
    valid = Y.loc[pre_years].notna().all(axis=0)
    for cov in covariates:
        valid = valid & cov_mats[cov].loc[pre_years].notna().all(axis=0)

    Y = Y.loc[:, valid]
    cov_mats = {cov: mat.loc[:, valid] for cov, mat in cov_mats.items()}

    if country not in Y.columns:
        raise ValueError(f"{country} not in columns after cleaning.")

    donors = [c for c in Y.columns if c != country]

    Y1 = Y[country]
    Y0 = Y[donors]

    # -----------------------
    # 2) BUILD MATCHING BLOCKS
    # -----------------------
    # Outcome path in pre-treatment period
    X0_pre_outcome = Y0.loc[pre_years].values          # T0 x J
    y1_pre_outcome = Y1.loc[pre_years].values          # T0

    # Covariate block: use pre-treatment means
    treated_cov_list = []
    donor_cov_list = []

    for cov in covariates:
        treated_mean = cov_mats[cov].loc[pre_years, country].mean()
        donor_means = cov_mats[cov].loc[pre_years, donors].mean(axis=0).values

        treated_cov_list.append(treated_mean)
        donor_cov_list.append(donor_means)

    if len(covariates) > 0:
        y1_cov = np.array(treated_cov_list)            # K
        X0_cov = np.vstack(donor_cov_list)             # K x J
    else:
        y1_cov = np.array([])
        X0_cov = np.empty((0, len(donors)))

    # Optional weighting: keeps outcome path more important if desired
    y1_pre_outcome = outcome_weight * y1_pre_outcome
    X0_pre_outcome = outcome_weight * X0_pre_outcome

    y1_cov = covariate_weight * y1_cov
    X0_cov = covariate_weight * X0_cov

    # Stack into one matching problem
    Z1 = np.concatenate([y1_pre_outcome, y1_cov])     # (T0 + K,)
    Z0 = np.vstack([X0_pre_outcome, X0_cov])          # (T0 + K) x J

    # -----------------------
    # 3) STANDARDIZE MATCHING ROWS
    #    critical when adding covariates
    # -----------------------
    row_means = Z0.mean(axis=1)
    row_stds = Z0.std(axis=1, ddof=0)
    row_stds[row_stds == 0] = 1.0

    Z0_scaled = (Z0 - row_means[:, None]) / row_stds[:, None]
    Z1_scaled = (Z1 - row_means) / row_stds

    Tmatch, J = Z0_scaled.shape

    # -----------------------
    # 4) SCM WITH COVARIATES
    # -----------------------
    w = cp.Variable(J)

    objective = cp.Minimize(cp.sum_squares(Z1_scaled - Z0_scaled @ w))
    constraints = [w >= 0, cp.sum(w) == 1]

    problem = cp.Problem(objective, constraints)
    problem.solve(solver=cp.OSQP)

    if w.value is None:
        raise RuntimeError("SCM optimization failed. Try solver=cp.SCS.")

    w_scm = np.array(w.value).ravel()
    weights_scm = pd.Series(w_scm, index=donors).sort_values(ascending=False)

    synthetic_scm = pd.Series(Y0.values @ w_scm, index=Y.index, name="Synthetic_SCM")

    # -----------------------
    # 5) RIDGE-ASCM
    # -----------------------
    pre_gap_match = Z1_scaled - Z0_scaled @ w_scm

    A = Z0_scaled @ Z0_scaled.T + ridge_lambda * np.eye(Tmatch)
    adjustment = Z0_scaled.T @ np.linalg.solve(A, pre_gap_match)

    w_ascm = w_scm + adjustment
    weights_ascm = pd.Series(w_ascm, index=donors).sort_values(ascending=False)

    synthetic_ascm = pd.Series(Y0.values @ w_ascm, index=Y.index, name="Synthetic_ASCM")

    # -----------------------
    # 6) EFFECTS + DIAGNOSTICS
    # -----------------------
    effect_scm = Y1 - synthetic_scm
    effect_ascm = Y1 - synthetic_ascm

    pre_rmse_scm = np.sqrt(np.mean((Y1.loc[pre_years] - synthetic_scm.loc[pre_years]) ** 2))
    pre_rmse_ascm = np.sqrt(np.mean((Y1.loc[pre_years] - synthetic_ascm.loc[pre_years]) ** 2))

    pre_sse_scm = np.sum((Y1.loc[pre_years] - synthetic_scm.loc[pre_years]) ** 2)
    pre_sse_ascm = np.sum((Y1.loc[pre_years] - synthetic_ascm.loc[pre_years]) ** 2)

    print(f"Pre-treatment RMSE (SCM):  {pre_rmse_scm:.4f}")
    print(f"Pre-treatment RMSE (ASCM): {pre_rmse_ascm:.4f}")

    print("\nTop SCM donor weights:")
    print(weights_scm.head(15))

    print("\nTop ASCM donor weights:")
    print(weights_ascm.head(15))

    print("\nMost negative ASCM weights:")
    print(weights_ascm.sort_values().head(10))

    return {
        "country": country,
        "treat_year": treat_year,
        "covariates": covariates,
        "ridge_lambda": ridge_lambda,
        "outcome_weight": outcome_weight,
        "covariate_weight": covariate_weight,
        "years": years,
        "pre_years": pre_years,
        "post_years": post_years,
        "Y": Y,
        "Y1": Y1,
        "Y0": Y0,
        "weights_scm": weights_scm,
        "weights_ascm": weights_ascm,
        "synthetic_scm": synthetic_scm,
        "synthetic_ascm": synthetic_ascm,
        "effect_scm": effect_scm,
        "effect_ascm": effect_ascm,
        "pre_rmse_scm": pre_rmse_scm,
        "pre_rmse_ascm": pre_rmse_ascm,
        "pre_sse_scm": pre_sse_scm,
        "pre_sse_ascm": pre_sse_ascm,
        "Z0_scaled": Z0_scaled,
        "Z1_scaled": Z1_scaled,
    }

In [70]:
covariates = ["emp","ctfp"]

brazil_ascm = scm_ascm(
    df=df,
    country="Brazil",
    treat_year=1996,
    covariates=covariates,
    ridge_lambda=1.0
)

Pre-treatment RMSE (SCM):  0.0236
Pre-treatment RMSE (ASCM): 0.0169

Top SCM donor weights:
Thailand         7.260067e-01
Botswana         2.739933e-01
Iran            -1.676820e-13
India           -2.041210e-13
Philippines     -3.566285e-13
Malaysia        -4.223766e-13
Nigeria         -4.779564e-13
Tanzania        -5.602582e-13
Israel          -6.974843e-13
Sri Lanka       -8.522699e-13
United States   -9.380374e-13
Egypt           -9.440200e-13
Hong Kong       -1.099246e-12
Luxembourg      -1.251909e-12
Japan           -1.431208e-12
dtype: float64

Top ASCM donor weights:
Thailand         0.831779
Botswana         0.246491
Singapore        0.122234
Germany          0.117769
Norway           0.117160
Japan            0.111087
Belgium          0.100011
South Korea      0.098953
Hong Kong        0.089012
India            0.078835
United States    0.057819
Switzerland      0.056812
Sri Lanka        0.049255
New Zealand      0.040743
Iran             0.040555
dtype: float64

Most negativ

In [71]:
covariates = ["emp","ctfp"]

usa_ascm = scm_ascm(
    df=df,
    country="United States",
    treat_year=2003,
    covariates=covariates,
    ridge_lambda=1.0
)

Pre-treatment RMSE (SCM):  0.0076
Pre-treatment RMSE (ASCM): 0.0049

Top SCM donor weights:
Luxembourg        4.315986e-01
India             2.750017e-01
Egypt             2.607661e-01
Brazil            3.263357e-02
Botswana          7.320617e-19
Philippines      -1.793532e-18
Iran             -2.458583e-18
Thailand         -2.737806e-18
Israel           -3.101562e-18
Hong Kong        -4.297982e-18
France           -5.670263e-18
United Kingdom   -6.788269e-18
Malaysia         -6.927869e-18
Sri Lanka        -7.095681e-18
Spain            -7.121038e-18
dtype: float64

Top ASCM donor weights:
Luxembourg     0.436892
India          0.308912
Egypt          0.260146
Australia      0.092538
Germany        0.086674
Japan          0.074769
Belgium        0.069548
Hong Kong      0.049422
Canada         0.047229
New Zealand    0.046019
Spain          0.039969
France         0.038750
Sri Lanka      0.033115
Israel         0.028619
Italy          0.026005
dtype: float64

Most negative ASCM weights: